<b> Estadística | El horizonte cambia la distribución. </b>  Simula 20_000 retornos diarios con una Student-\(t\), df=4, y escala a aproximadamente 1% de volatilidad diaria. Construye retornos acumulados no solapados a horizontes 
$ (H=\{1,5,20\})$:

$$ R_{t,H}=\prod_{j=0}^{H-1}(1+r_{t+j})-1. $$

Para cada horizonte calcula solamente std, skew, kurtosis y P(R_H < -2*std_H). 
No anualices. 

La pregunta es: ¿la distribución a 20 días es simplemente “la distribución de un día multiplicada por 20”? Explica qué cambió al agregar retornos y qué propiedades todavía podrían impedir una aproximación normal razonable en datos financieros reales.

In [9]:
import numpy as np 
import scipy.stats as stats
import pandas as pd 
# generamos serie de retornos usando una distribucion t-student con 4 grados de libertad 
# generador reproducible 
rng = np.random.default_rng(seed = 42)
retornos_diarios = rng.standard_t(df = 4, size = 20000)
# volatilidad diaria : 1% , escalar la serie  
vol_1d = 0.01 
retornos_diarios = (retornos_diarios / retornos_diarios.std(ddof = 1)) * vol_1d
horizontes = [1,5,20]
retornos_acumulados = {}
for h in horizontes:
    bloques = retornos_diarios.reshape(-1,h)
    # retornos acumulados no zolapados para 1dia,5, 20 
    retornos_h = np.prod(1+bloques, axis = 1) - 1
    retornos_acumulados[f'{h}d'] =retornos_h

def metricas(retornos):
    std_h = retornos.std(ddof=1)
    return {'std' : std_h, 
            'skew': stats.skew(retornos),
            'kurtosis': stats.kurtosis(retornos), 
            'P(R_H < -2*std_H)': (retornos < -2 * std_h).mean()}
resultados = []
for horizonte, retornos in retornos_acumulados.items(): 
    fila = {'horizontes': horizonte}
    fila.update(metricas(retornos))
    resultados.append(fila)
df_metricas = pd.DataFrame(resultados)
df_metricas

,horizontes,std,skew,kurtosis,P(R_H < -2*std_H)
0,1d,0.010000,0.953190,25.518796,0.0234
1,5d,0.022188,0.518217,5.347729,0.0215
2,20d,0.044496,0.127074,0.985706,0.0240
